# 市场环境综合评估Dashboard

**目标**: 一站式展示所有市场环境评估指标

**包含**:
- 8个动态参数接口
- 多周期共振
- IBD反转信号
- 风险预警
- 操作建议

In [ ]:
# 统一环境初始化（自动检测项目路径）from notebooks.lib import (    setup_research_environment,    ErrorBoundary,    ResultSaver)# 初始化研究环境env = setup_research_environment(verbose=True)# 导入必要的库import pandas as pdimport numpy as npfrom datetime import datetime, timedelta# 从环境获取组件jq = Nonetry:    jq = env.get_jqdata_client()except Exception as e:    print(f"⚠️ JQData初始化失败: {{e}}")# 加载配置config = env.load_config('config')INDEX_CODE = config.get('data', {{}}).get('default_index', '000001.XSHG')# 初始化结果保存器result_saver = ResultSaver("comprehensive_dashboard")print('✅ 环境加载完成')

## 执行综合评估

In [ ]:
index_code = "000001.XSHG"
eval_result = evaluator.evaluate(index_code=index_code)
signals = signal_provider.get_all_signals(index_code=index_code)
fusion_result = signal_fusion.fuse(eval_result, method=FusionMethod.WEIGHTED_AVERAGE)

print(f"✅ 评估完成: {eval_result.evaluation_date}")

## 1. 核心指标仪表盘

In [ ]:
def create_dashboard_html(signals, eval_result):
    # 颜色映射
    def get_color(value, positive_is_good=True):
        if positive_is_good:
            if value > 0.3: return '#27ae60'
            elif value > 0: return '#f39c12'
            else: return '#e74c3c'
        else:
            if value < 30: return '#27ae60'
            elif value < 60: return '#f39c12'
            else: return '#e74c3c'
    
    html = f'''
    <div style="font-family: Arial, sans-serif; padding: 20px; background: #f8f9fa;">
        <h2 style="text-align: center; color: #2c3e50;">📊 市场环境评估Dashboard</h2>
        <p style="text-align: center; color: #7f8c8d;">评估日期: {signals.evaluation_date} | 指数: {signals.index_code}</p>
        
        <div style="display: flex; flex-wrap: wrap; justify-content: space-around; margin-top: 20px;">
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 200px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">趋势得分</h4>
                <p style="font-size: 28px; margin: 10px 0; color: {get_color(signals.trend_score)};">{signals.trend_score:.3f}</p>
            </div>
            
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 200px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">市场环境</h4>
                <p style="font-size: 28px; margin: 10px 0; color: #2c3e50;">{signals.market_regime.upper()}</p>
            </div>
            
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 200px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">建议仓位</h4>
                <p style="font-size: 28px; margin: 10px 0; color: #3498db;">{signals.suggested_position_ratio:.1%}</p>
            </div>
            
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 200px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">风险得分</h4>
                <p style="font-size: 28px; margin: 10px 0; color: {get_color(signals.risk_exposure_score, False)};">{signals.risk_exposure_score:.1f}/100</p>
            </div>
        </div>
        
        <div style="display: flex; flex-wrap: wrap; justify-content: space-around; margin-top: 10px;">
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 150px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">反转信号</h4>
                <p style="font-size: 20px; margin: 10px 0; color: {get_color(signals.reversal_signal)};">{signals.reversal_signal:.3f}</p>
            </div>
            
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 150px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">风格建议</h4>
                <p style="font-size: 20px; margin: 10px 0; color: #9b59b6;">{signals.allocation_style_shift}</p>
            </div>
            
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 150px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">波动环境</h4>
                <p style="font-size: 20px; margin: 10px 0; color: #e67e22;">{signals.volatility_regime}</p>
            </div>
            
            <div style="background: white; padding: 15px; margin: 10px; border-radius: 8px; min-width: 150px; box-shadow: 0 2px 4px rgba(0,0,0,0.1);">
                <h4 style="margin: 0; color: #7f8c8d;">交易频率</h4>
                <p style="font-size: 20px; margin: 10px 0; color: #1abc9c;">{signals.trade_frequency_suggestion}</p>
            </div>
        </div>
    </div>
    '''
    return html

display(HTML(create_dashboard_html(signals, eval_result)))

## 2. 详细分析报告

In [ ]:
print("="*70)
print("📈 趋势分析 (TrendAnalyzer)")
print("="*70)
if eval_result.trend_result:
    tr = eval_result.trend_result
    print(f"综合得分: {tr.composite_score:.2f}")
    print(f"市场阶段: {tr.market_phase}")
    print(f"短期: {tr.short_term.score:.1f} | 中期: {tr.medium_term.score:.1f} | 长期: {tr.long_term.score:.1f}")

In [ ]:
print("\n" + "="*70)
print("🏛️ 市场环境 (MarketRegimeDetector)")
print("="*70)
if eval_result.regime_result:
    rr = eval_result.regime_result
    print(f"市场环境: {rr.regime.value}")
    print(f"综合得分: {rr.score:.2f}")
    print(f"置信度: {rr.confidence:.2%}")

In [ ]:
print("\n" + "="*70)
print("📉 IBD反转信号 (IBDStyleAnalyzer)")
print("="*70)
if eval_result.ibd_result:
    ir = eval_result.ibd_result
    print(f"市场状态: {ir.market_status.value}")
    print(f"分布日: {ir.distribution_count}个 | 跟踪日: {len(ir.follow_through_days)}个")
    print(f"反转信号强度: {eval_result.reversal_signal:.3f}")
    if ir.recommendation:
        print(f"建议: {ir.recommendation}")

In [ ]:
print("\n" + "="*70)
print("🔀 信号融合 (SignalFusion)")
print("="*70)
print(f"融合方法: {fusion_result.method}")
print(f"融合趋势得分: {fusion_result.fused_trend_score:.3f}")
print(f"融合市场环境: {fusion_result.fused_market_regime}")
print(f"信号一致性: {fusion_result.consistency:.2%}")
print(f"置信度: {fusion_result.confidence:.2%}")

## 3. 综合可视化

In [ ]:
fig = plt.figure(figsize=(16, 10))

# 雷达图数据
categories = ['趋势得分', '反转信号', '仓位建议', '风险(反向)', '一致性', '置信度']
values = [
    (signals.trend_score + 1) / 2,  # 归一化到0-1
    (signals.reversal_signal + 1) / 2,
    signals.suggested_position_ratio,
    1 - signals.risk_exposure_score / 100,  # 风险反向
    fusion_result.consistency,
    fusion_result.confidence
]

# 雷达图
ax1 = fig.add_subplot(221, projection='polar')
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
values_plot = values + [values[0]]
angles += angles[:1]

ax1.plot(angles, values_plot, 'o-', linewidth=2, color='blue')
ax1.fill(angles, values_plot, alpha=0.25, color='blue')
ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories)
ax1.set_title('综合指标雷达图')

# 8信号条形图
ax2 = fig.add_subplot(222)
signal_names = ['趋势', '反转', '仓位', '风格', '风险', '波动', '频率']
signal_values = [
    signals.trend_score,
    signals.reversal_signal,
    signals.suggested_position_ratio,
    0.7 if signals.allocation_style_shift == 'growth' else 0.3,
    signals.risk_exposure_score / 100,
    {'low': 0.2, 'medium': 0.5, 'high': 0.8, 'extreme': 1.0}.get(signals.volatility_regime, 0.5),
    {'monthly': 0.2, 'weekly': 0.4, 'daily': 0.6, 'intraday': 0.8}.get(signals.trade_frequency_suggestion, 0.5)
]
colors = ['green' if v > 0.5 else 'orange' if v > 0.3 else 'red' for v in signal_values]
ax2.barh(signal_names, signal_values, color=colors, alpha=0.7)
ax2.set_xlim(-1, 1)
ax2.axvline(x=0, color='black', linestyle='--')
ax2.set_title('8个动态信号概览')

# 三周期趋势
ax3 = fig.add_subplot(223)
if eval_result.trend_result:
    periods = ['短期', '中期', '长期']
    scores = [
        eval_result.trend_result.short_term.score,
        eval_result.trend_result.medium_term.score,
        eval_result.trend_result.long_term.score
    ]
    colors = ['green' if s > 0 else 'red' for s in scores]
    ax3.bar(periods, scores, color=colors, alpha=0.7)
    ax3.axhline(y=0, color='black', linestyle='--')
    ax3.set_ylim(-100, 100)
ax3.set_title('多周期趋势得分')

# 风险分解饼图
ax4 = fig.add_subplot(224)
risk_labels = ['趋势风险', '环境风险', '波动风险', '其他']
risk_values = [20, 30, 25, 25]  # 示例值
ax4.pie(risk_values, labels=risk_labels, autopct='%1.1f%%', colors=['#e74c3c', '#f39c12', '#9b59b6', '#95a5a6'])
ax4.set_title('风险来源分解')

plt.tight_layout()
plt.show()

## 4. 操作建议汇总

In [ ]:
print("\n" + "="*70)
print("💡 操作建议汇总")
print("="*70)

# 仓位建议
print(f"\n1. 仓位管理")
print(f"   建议仓位: {signals.suggested_position_ratio:.0%}")
if signals.suggested_position_ratio >= 0.7:
    print("   策略: 进攻型配置")
elif signals.suggested_position_ratio >= 0.5:
    print("   策略: 稳健型配置")
else:
    print("   策略: 防御型配置")

# 风格建议
print(f"\n2. 风格配置")
print(f"   建议风格: {signals.allocation_style_shift}")
style_map = {
    'growth': '偏好成长股、科技股',
    'value': '偏好价值股、红利股',
    'balanced': '均衡配置',
    'defensive': '防御型配置，关注低波动标的'
}
print(f"   说明: {style_map.get(signals.allocation_style_shift, '')}")

# 交易频率
print(f"\n3. 交易频率")
print(f"   建议频率: {signals.trade_frequency_suggestion}")
freq_map = {
    'intraday': '可考虑日内交易',
    'daily': '日频调仓',
    'weekly': '周频调仓',
    'monthly': '月频调仓，持仓为主'
}
print(f"   说明: {freq_map.get(signals.trade_frequency_suggestion, '')}")

# 风险提示
print(f"\n4. 风险提示")
print(f"   风险得分: {signals.risk_exposure_score:.0f}/100")
if signals.risk_exposure_score >= 70:
    print("   ⚠️ 风险较高，请谨慎操作")
elif signals.risk_exposure_score >= 50:
    print("   ⚡ 风险适中，注意止损")
else:
    print("   ✅ 风险可控")

## 5. 保存综合报告

In [ ]:
comprehensive_report = {
    "evaluation_date": signals.evaluation_date,
    "index_code": signals.index_code,
    "dynamic_signals": signals.to_dict(),
    "fusion_result": {
        "fused_trend_score": fusion_result.fused_trend_score,
        "fused_market_regime": fusion_result.fused_market_regime,
        "consistency": fusion_result.consistency,
        "confidence": fusion_result.confidence
    },
    "recommendations": {
        "position": signals.suggested_position_ratio,
        "style": signals.allocation_style_shift,
        "frequency": signals.trade_frequency_suggestion,
        "risk_level": "high" if signals.risk_exposure_score >= 70 else "medium" if signals.risk_exposure_score >= 50 else "low"
    }
}

save_research_conclusion(
    module="comprehensive_dashboard",
    findings=comprehensive_report,
    recommendation=f"市场: {signals.market_regime}, 仓位: {signals.suggested_position_ratio:.0%}, 风格: {signals.allocation_style_shift}",
    metadata={"index_code": index_code}
)

print("\n✅ 综合报告已保存")
print(f"📅 评估时间: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")